# pyrepl

> A Python prompt you own, and an agent in the layer above it.

The session machinery behind `ramabana --python`: a private kernel, a host whose Python tools
go through Dhrishti's protected overlay, and the small judgements a Python prompt needs. The
prompt itself is `cli.Ui`, because there is one terminal and python is a mode inside it.

The namespace belongs to whoever is typing; the agent reads it and builds in a layer of its
own, and cannot rebind a name it did not create. That guarantee is not this module's: Dhrishti
serves the kernel's namespace and splits its API in two, and all anything here does is point
the agent at the half that cannot mutate anything.

In [ ]:
#| default_exp pyrepl

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import test_eq, test_fail
import threading

In [ ]:
#| export
import asyncio, codeop, json, os, queue, re, shutil, sys, tempfile, urllib.parse, urllib.request
from dataclasses import dataclass, field
from pathlib import Path
from rich.text import Text
from ramabana.core import agent_err
from ramabana.tools import LocalHost

## Outputs

A kernel reports its results as a stream of messages; a notebook stores them as a list of
dicts. `ExecOutcome` is one request in the notebook's shape, which is what lets the same
outputs go to the terminal, to `log_cell` and to a test without a second representation.

In [ ]:
#| export
@dataclass
class ExecOutcome:
    "One kernel request in nbformat's output shape."
    ok: bool = True
    outputs: list = field(default_factory=list)
    execution_count: int | None = None
    error: str | None = None

def output_text(outputs):
    "Flatten notebook outputs for tests, logs and plain terminal fallbacks."
    parts = []
    for out in outputs:
        kind = out.get('output_type')
        if kind == 'stream': parts.append(_text(out.get('text')))
        elif kind in ('execute_result', 'display_data'):
            data = out.get('data') or {}
            parts.append(_text(data.get('text/plain') or data.get('text/markdown')))
        elif kind == 'error':
            trace = out.get('traceback') or []
            parts.append('\n'.join(trace) if trace else f"{out.get('ename')}: {out.get('evalue')}")
    return '\n'.join(p.rstrip('\n') for p in parts)

def _text(value):
    return ''.join(value) if isinstance(value, list) else str(value or '')

In [ ]:
outs = [{'output_type': 'stream', 'text': 'hello\n'},
        {'output_type': 'execute_result', 'data': {'text/plain': '42'}},
        {'output_type': 'error', 'ename': 'ValueError', 'evalue': 'bad', 'traceback': []}]
test_eq(output_text(outs), 'hello\n42\nValueError: bad')
# A stream arrives split across messages, so the list form has to flatten rather than repr.
test_eq(output_text([{'output_type': 'stream', 'text': ['a', 'b']}]), 'ab')
# An error with a traceback prefers it: the ename alone loses where it happened.
test_eq(output_text([{'output_type': 'error', 'ename': 'E', 'evalue': 'v',
                      'traceback': ['line one', 'line two']}]), 'line one\nline two')
test_eq(output_text([]), '')

## The kernel

A private `ipykernel`, and Dhrishti started *inside* it -- because the namespace Dhrishti
serves is the kernel's, and because its server runs on a background thread, which is what
keeps it answering during exactly the long cell you most want to watch.

The port comes back through a printed marker rather than a return value: the bootstrap runs
as a cell, and a cell's only channel to the caller is its output.

In [ ]:
#| export
class Kernel:
    "A private ipykernel with Dhrishti serving its live namespace."
    def __init__(self, cwd='.'):
        self.cwd = Path(cwd).resolve()
        self.km = self.kc = None
        self.base = None
        self._ipc_dir = None
        self._exec_lock = asyncio.Lock()
        self._shell_lock = asyncio.Lock()

    @property
    def alive(self):
        return self.km is not None and self.kc is not None and self.km.has_kernel

    async def start(self, timeout=60):
        from jupyter_client.kernelspec import KernelSpec
        from jupyter_client.manager import AsyncKernelManager
        opts = {'kernel_name': 'python3'}
        if os.name != 'nt':
            self._ipc_dir = tempfile.mkdtemp(prefix='rama-k-', dir='/tmp' if os.path.isdir('/tmp') else None)
            opts.update(transport='ipc', ip=os.path.join(self._ipc_dir, 'k'))
        self.km = AsyncKernelManager(**opts)
        self.km._kernel_spec = KernelSpec(
            argv=[sys.executable, '-m', 'ipykernel_launcher', '-f', '{connection_file}'],
            display_name='Ramabana PyREPL', language='python')
        try:
            await self.km.start_kernel(cwd=str(self.cwd))
            self.kc = self.km.client()
            self.kc.start_channels()
            await self.kc.wait_for_ready(timeout=timeout)
            await self._bootstrap()
            return self
        except BaseException:
            await self.shutdown()
            raise

    async def _bootstrap(self):
        root = self.cwd/'.ramabana'/'pyrepl'
        # '<project>-pyrepl-<pid>', sanitised because --attach is where a human types it
        proj = re.sub(r'[^a-z0-9]+', '-', (self.cwd.name or 'ramabana').lower()).strip('-') or 'ramabana'
        source = (
            'def _ramabana_bootstrap():\n'
            ' import dhrishti.serving as ds, os\n'
            f' p = ds.serve_in_kernel(name={proj!r} + "-pyrepl-" + str(os.getpid()), agent="restricted", token=True, '
            f'session_dir={str(root/"sessions")!r}, agent_session_dir={str(root/"agents")!r})\n'
            ' print("__RAMABANA_DHRISHTI__" + str(p))\n'
            '_ramabana_bootstrap()\n'
            'del _ramabana_bootstrap')
        result = await self.execute(source, store_history=False)
        marker = next((line for out in result.outputs if out.get('output_type') == 'stream'
                       for line in _text(out.get('text')).splitlines()
                       if line.startswith('__RAMABANA_DHRISHTI__')), '')
        if not result.ok or not marker:
            raise RuntimeError(result.error or output_text(result.outputs) or 'Dhrishti did not start')
        self.base = f'http://127.0.0.1:{int(marker.removeprefix("__RAMABANA_DHRISHTI__"))}'

    async def execute(self, code, store_history=True, on_output=None):
        "Execute one cell and stream each nbformat-shaped output to `on_output`."
        if not self.alive: return ExecOutcome(ok=False, error='kernel is not running')
        async with self._exec_lock, self._shell_lock:
            msg_id = self.kc.execute(str(code), store_history=store_history, allow_stdin=False)
            outputs = await self._collect(msg_id, on_output)
            reply = await self._reply(msg_id)
        content = (reply or {}).get('content') or {}
        error = None
        if content.get('status') == 'error': error = f"{content.get('ename')}: {content.get('evalue')}"
        elif content.get('status') == 'abort': error = 'aborted'
        if error is None:
            err = next((o for o in outputs if o.get('output_type') == 'error'), None)
            if err: error = f"{err.get('ename')}: {err.get('evalue')}"
        return ExecOutcome(error is None, outputs, content.get('execution_count'), error)

    async def _collect(self, msg_id, on_output):
        outputs, displays = [], {}
        while True:
            try: message = await self.kc.get_iopub_msg(timeout=.5)
            except (queue.Empty, asyncio.TimeoutError):
                if not self.alive: break
                continue
            if (message.get('parent_header') or {}).get('msg_id') != msg_id: continue
            kind, content = message['header']['msg_type'], message['content']
            if kind == 'status' and content.get('execution_state') == 'idle': break
            if kind == 'clear_output': outputs.clear(); displays.clear(); continue
            out = self._output(kind, content)
            if out is None: continue
            display_id = (content.get('transient') or {}).get('display_id')
            if kind == 'update_display_data' and display_id in displays:
                outputs[displays[display_id]] = out
            elif (out['output_type'] == 'stream' and outputs and outputs[-1].get('output_type') == 'stream'
                  and outputs[-1].get('name') == out.get('name')):
                outputs[-1]['text'] += out['text']
            else:
                outputs.append(out)
                if display_id: displays[display_id] = len(outputs) - 1
            if on_output: on_output(out)
        return outputs

    @staticmethod
    def _output(kind, content):
        if kind == 'stream':
            return {'output_type': 'stream', 'name': content.get('name', 'stdout'), 'text': content.get('text', '')}
        if kind == 'error':
            return {'output_type': 'error', 'ename': content.get('ename', ''),
                    'evalue': content.get('evalue', ''), 'traceback': list(content.get('traceback') or [])}
        if kind in ('execute_result', 'display_data', 'update_display_data'):
            return {'output_type': 'execute_result' if kind == 'execute_result' else 'display_data',
                    'data': content.get('data') or {}, 'metadata': content.get('metadata') or {}}
        return None

    async def _reply(self, msg_id):
        while True:
            try: message = await self.kc.get_shell_msg(timeout=10)
            except (queue.Empty, asyncio.TimeoutError): return None
            if (message.get('parent_header') or {}).get('msg_id') == msg_id: return message

    async def complete(self, code, pos):
        if not self.alive: return [], int(pos)
        async with self._shell_lock:
            msg_id = self.kc.complete(str(code), int(pos))
            while True:
                message = await self.kc.get_shell_msg(timeout=10)
                if (message.get('parent_header') or {}).get('msg_id') == msg_id:
                    content = message.get('content') or {}
                    return content.get('matches') or [], int(content.get('cursor_start', pos))

    async def interrupt(self):
        if self.km: await self.km.interrupt_kernel()

    async def shutdown(self, timeout=5):
        if self.kc:
            try: self.kc.stop_channels()
            except Exception: pass
        if self.km and self.km.has_kernel:
            # a dhrishti kernel may never agree to `now=False`, so the ask has a deadline
            try: await asyncio.wait_for(self.km.shutdown_kernel(now=False), timeout)
            except Exception:
                try: await self.km.shutdown_kernel(now=True)
                except Exception: pass
        self.km = self.kc = self.base = None
        if self._ipc_dir:
            shutil.rmtree(self._ipc_dir, ignore_errors=True)
            self._ipc_dir = None

In [ ]:
kernel = await Kernel('.').start()
r = await kernel.execute('a = 6 * 7\na')
test_eq((r.ok, output_text(r.outputs)), (True, '42'))
assert kernel.base.startswith('http://127.0.0.1:')     # dhrishti came up inside the kernel

In [ ]:
# A dead kernel answers rather than hanging on a channel nobody is writing to.
dead = Kernel('.')
r = await dead.execute('1 + 1')
test_eq((r.ok, r.error), (False, 'kernel is not running'))

# An error is an outcome, not an exception: the caller is a UI.
r = await kernel.execute('1/0')
test_eq(r.ok, False)
assert 'ZeroDivisionError' in r.error and 'ZeroDivisionError' in output_text(r.outputs)

# Streams are coalesced, so a loop printing forty lines is one output.
r = await kernel.execute("for i in range(3): print(i)")
test_eq(len([o for o in r.outputs if o['output_type'] == 'stream']), 1)
test_eq(output_text(r.outputs), '0\n1\n2')

# Completion comes from the kernel, which is IPython's completer with jedi behind it.
await kernel.execute('import json')
matches, start = await kernel.complete('json.du', 7)
test_eq((sorted(matches), start), (['dump', 'dumps'], 5))   # the attribute, and where it starts

# And a dead kernel answers here too: `shutdown` drops `kc`.
test_eq(await dead.complete('json.du', 7), ([], 7))

## The host

An ordinary Ramabana host -- the file, search and web tools are the same ones -- whose *Python*
tools go over HTTP to the agent half of the Dhrishti API. That half is the ungated one, and
that is the whole protection: the token that opens `/api/exec`, `/api/set` and `/api/promote`
is never read here, so there is no code path from a tool call to the owner's namespace.

`log_cell` is what makes the session one artifact. The owner's cells go in as code with their
real outputs, each turn as `**user**`/`**assistant**` markdown, in the order they happened.

In [ ]:
#| export
def _api(base, path, params=None, timeout=60):
    query = urllib.parse.urlencode(params or {})
    url = base.rstrip('/') + path + (('?' + query) if query else '')
    with urllib.request.urlopen(url, timeout=timeout) as response:
        return json.loads(response.read())

class DhrishtiHost(LocalHost):
    "A project host whose Python tools use a protected Dhrishti overlay."
    def __init__(self, roots, base, **kwargs):
        super().__init__(roots, **kwargs)
        self.base = base
        try: info = _api(base, '/agent/api/info')
        except Exception: info = {}
        log = (info or {}).get('log')
        self.agent_log = Path(log) if log else None   # None, not `Path('')`, which is truthy

    def run_python(self, code):
        return self.inspect_python(code, 'overlay')   # always the overlay; scope is model-settable

    def inspect_python(self, code, scope='isolated'):
        if scope not in self.scopes: return f'this host only honours {self.scopes}'
        try: result = _api(self.base, '/agent/api/exec', {'code': str(code), 'scope': scope})
        except Exception as exc: return agent_err(exc)
        if result.get('error'): return str(result['error'])
        out = result.get('stdout') or ''
        value = result.get('result')
        if isinstance(value, dict): value = value.get('value')
        if value is not None: out += ('\n' if out else '') + str(value)
        return out or '(ok)'

    @property
    def scopes(self): return ('isolated', 'overlay')

    @property
    def kernel_kind(self): return 'ipykernel'

    def list_vars(self):
        try: result = _api(self.base, '/agent/api/rows', {'profile': 'minimal', 'sort': 'name'})
        except Exception as exc: return agent_err(exc)
        return '\n'.join(f"{node.get('name')}: {node.get('type')} = {node.get('value')}"
                         for group in result.get('groups', []) for node in group.get('nodes', []))

    def log_cell(self, source, outputs=None, cell_type='code'):
        "Append a human Python or model markdown turn to Dhrishti's session notebook."
        # read/append/write per cell: the owner's kernel writes this file too
        if self.agent_log is None: return
        from fastcore.nbio import read_nb, write_nb, new_nb, mk_cell
        self.agent_log.parent.mkdir(parents=True, exist_ok=True)
        nb = read_nb(self.agent_log) if self.agent_log.exists() else new_nb([])
        cell = mk_cell(str(source), cell_type)
        if cell_type == 'code':
            cell['outputs'], cell['execution_count'] = list(outputs or []), None
        nb.cells.append(cell)
        write_nb(nb, self.agent_log)

    def describe(self):
        "name -> 'type [shape]' for everything the session is holding. Never raises."
        try: result = _api(self.base, '/agent/api/rows', {'profile': 'minimal', 'sort': 'name'}, timeout=2)
        except Exception: return {}
        out = {}
        for group in result.get('groups', []):
            for node in group.get('nodes', []):
                if name := node.get('name'):
                    value = node.get('value')   # falsy is still a value; only a missing key is not
                    out[name] = str(value if value is not None else (node.get('type') or ''))
        return out

In [ ]:
host = DhrishtiHost(['.'], kernel.base, web=False, index=False)
await kernel.execute("owner = {'kept': 1}")

test_eq(host.run_python('mine = len(owner)'), '(ok)')      # reads the owner freely
assert 'mine' in host.list_vars() and 'owner' in host.list_vars()

# The write lands in the agent's layer and the owner's binding is untouched.
host.run_python('owner = None')
r = await kernel.execute('owner')
test_eq(output_text(r.outputs), "{'kept': 1}")

# A transport that is not there comes back as a sentence, because a tool cannot raise usefully.
assert 'Error' in DhrishtiHost(['.'], 'http://127.0.0.1:1', web=False, index=False).run_python('1')

In [ ]:
from fastcore.nbio import read_nb
r = await kernel.execute('logged = 1')
host.log_cell('logged = 1', r.outputs)
host.log_cell('**user**\n\nwhat is logged?', cell_type='markdown')
nb = read_nb(host.agent_log)
test_eq([c.cell_type for c in nb.cells[-2:]], ['code', 'markdown'])
test_eq(nb.cells[-2].outputs, r.outputs)          # the real outputs, not a rendering of them

In [ ]:
# Both ways to an absent log path: dhrishti answers `'log': None`, and a base that is not there
# leaves the constructor's `except` with the same empty info.
_real_api = _api
_api = lambda base, path, params=None, timeout=60: {'log': None}
try: nolog = DhrishtiHost(['.'], kernel.base, web=False, index=False)
finally: _api = _real_api
dead_host = DhrishtiHost(['.'], 'http://127.0.0.1:1', web=False, index=False)
test_eq((nolog.agent_log, dead_host.agent_log), (None, None))
test_eq([h.log_cell('x = 1', []) for h in (nolog, dead_host)], [None, None])   # logs nothing, quietly

In [ ]:
# `describe`'s fallback, against a canned `_api`: dhrishti's `value_str` always reprs a real
# value to a non-empty string, so only a raw-JSON transport reaches the falsy branch.
_real_api = _api
def _fake_rows_api(base, path, params=None, timeout=60):
    return {'groups': [{'nodes': [
        {'name': 'zero_val', 'type': 'int', 'value': 0},
        {'name': 'blank_val', 'type': 'str', 'value': ''},
        {'name': 'missing_val', 'type': 'NoneType', 'value': None},
    ]}]}
_api = _fake_rows_api
try: described = host.describe()
finally: _api = _real_api
test_eq(described['zero_val'], '0')          # falsy but present: the value, not the type
test_eq(described['blank_val'], '')          # same for an empty string
test_eq(described['missing_val'], 'NoneType')  # genuinely absent: falls back to the type

### Completion, with the namespace described

Two sources, and the split is the point. **The kernel completes** -- `complete_request` is
IPython's completer with jedi behind it, so it knows imports, attribute chains, dict keys and
paths. Dhrishti has no completion endpoint at all; its API is rows, expand, grid, result,
history, sessions, and rows only knows top-level bindings, so a completer built on it would
be a downgrade.

**Dhrishti describes.** Each candidate that names something live gains its type and shape, so
the row reads `df → DataFrame [1200×8]` rather than `df`. Describing a namespace needs no
token, which is why it comes from the agent half of the API.

Best-effort, always: the list paints from the kernel's answer and gains descriptions if they
arrive. A slow or broken lookup leaves bare names, never an empty list.

In [ ]:
#| export
def annotate(matches, described):
    "Candidate names with their type and shape where the session knows one."
    # exact names only: `df.copy` is a method on a binding, not a binding
    return [f'{m} -> {described[m]}' if m in described else m for m in matches]

In [ ]:
test_eq(annotate(['df', 'dfs'], {'df': 'DataFrame [3×2]'}), ['df -> DataFrame [3×2]', 'dfs'])
test_eq(annotate(['x'], {}), ['x'])
test_eq(annotate([], {'x': 'int'}), [])
# An attribute chain is not a binding, so it stays bare rather than being mislabelled.
test_eq(annotate(['df.copy'], {'df': 'DataFrame [3×2]'}), ['df.copy'])

## Locking into a session someone else owns

Inside leela the kernel already exists and already has an agent session, so `ramabana --attach
NAME` starts nothing: it finds the running server and builds the host against it. Leela owns the
human's prompt, so there is no python mode; Ramabana is the agent beside it.

Discovery goes through Dhrishti's registry, and **not** by calling `serve_in_kernel` again.
`_start` is guarded by `if _server is None`, but the lines above that guard are not: `_profile`,
`_env` and `_agent_mode` are reassigned unconditionally, and passing `session_dir` or
`agent_session_dir` calls `set_logging` regardless. Re-serving inside leela's kernel would hand
back the right port while repointing leela's session logs and resetting its agent mode -- a
`readonly` session would come back `restricted`. The registry is read-only, so it cannot.

Promotion is not Ramabana's to perform here. The agent's work is visible in the shared session
notebook and over the same API leela is already watching, and the human promotes from the
surface that holds the token.

In [ ]:
#| export
def _ambiguous(attach, cands):
    "Error text for a name that matches more than one live session -- list them, do not pick."
    rows = '; '.join(f"{e.get('name')} (cwd={e.get('cwd')}, port={e.get('port')})" for e in cands)
    return f'{attach!r} matches more than one live dhrishti session, refusing to guess which: {rows}'

def find_session(attach):
    "The base URL of a live Dhrishti session named `attach`, or `attach` itself if it is a URL."
    attach = str(attach).strip()
    if '://' in attach: return attach
    from dhrishti.serving import active   # the registry, not `serve_in_kernel`, which would re-serve
    entries = active()
    exact = [e for e in entries if e.get('name') == attach]
    if len(exact) > 1: raise RuntimeError(_ambiguous(attach, exact))
    hit = exact[0] if exact else None
    # a name that is not an exact match is tried as a project prefix, anchored at '-pyrepl-'
    if hit is None and attach:
        prefixed = [e for e in entries if str(e.get('name') or '').startswith(attach + '-pyrepl-')]
        if len(prefixed) > 1: raise RuntimeError(_ambiguous(attach, prefixed))
        hit = prefixed[0] if prefixed else None
    if hit is None and attach:
        # The whole basename, not a substring, so 'leela' does not also match 'old-leela-backup'.
        by_cwd = [e for e in entries if Path(str(e.get('cwd') or '')).name == attach]
        if len(by_cwd) > 1: raise RuntimeError(_ambiguous(attach, by_cwd))
        hit = by_cwd[0] if by_cwd else None
    if hit is None: raise RuntimeError(
        f'no live dhrishti session matching {attach!r}; running: '
        f'{", ".join(str(e.get("name")) for e in entries) or "none"}')
    return hit.get('base') or f'http://127.0.0.1:{hit["port"]}'


def sessions():
    "Every live Dhrishti session: what to pass to `--attach`, and where it is."
    from dhrishti.serving import active
    rows = [(str(e.get('name') or '?'), str(e.get('cwd') or '?'), int(e.get('port') or 0),
             e.get('base') or f"http://127.0.0.1:{e.get('port')}") for e in active()]
    if not rows: return 'no live sessions; start one with `ramabana --python`'
    dupes = {n for n, *_ in rows if sum(1 for m, *_ in rows if m == n) > 1}
    out = []
    for name, cwd, port, base in sorted(rows):
        # a name shared by two live sessions cannot resolve, so show what does
        out.append(f'{base if name in dupes else name:<34} {cwd}  :{port}')
    return '\n'.join(out)

In [ ]:
# Anything with a scheme is taken as given, so an explicit URL never needs a registry.
test_eq(find_session('http://127.0.0.1:9999'), 'http://127.0.0.1:9999')
test_fail(lambda: find_session('no-such-session-anywhere'), contains='no live dhrishti session')

`find_session`'s resolution logic is pinned with synthetic registry entries rather than the
machine's real one, which is shared with every other dhrishti session on this box.
`unittest.mock.patch` swaps `dhrishti.serving.active` for the duration of each assertion;
`find_session` still runs its real resolution code against it.

In [ ]:
from unittest.mock import patch

def _entry(name, port, cwd='/x/proj'):
    return {'name': name, 'port': port, 'pid': port, 'cwd': cwd, 'base': f'http://127.0.0.1:{port}', 'started': 0}

one = [_entry('proj-pyrepl-111', 9001, '/home/proj')]
dupe_name = [_entry('proj-pyrepl-111', 9001, '/home/a'), _entry('proj-pyrepl-111', 9002, '/home/b')]
by_cwd = [_entry('some-other-name-222', 9003, '/home/leela')]

with patch('dhrishti.serving.active', lambda: one):
    # An exact, unique name resolves straight to its base.
    test_eq(find_session('proj-pyrepl-111'), 'http://127.0.0.1:9001')
    # A project prefix resolves too, when it picks out exactly one: a human types 'proj'.
    test_eq(find_session('proj'), 'http://127.0.0.1:9001')

with patch('dhrishti.serving.active', lambda: dupe_name):
    # Two live entries sharing a name are refused rather than guessed at, and both are named.
    try:
        find_session('proj-pyrepl-111')
        assert False, 'expected a refusal'
    except RuntimeError as e:
        msg = str(e)
        assert 'matches more than one' in msg
        assert '/home/a' in msg and '/home/b' in msg   # both candidates identifiable, not just counted
        assert '9001' in msg and '9002' in msg

cross_project = [_entry('ramabana-pyrepl-1', 9004, '/home/ramabana'),
                 _entry('ramabana-extra-pyrepl-2', 9005, '/home/ramabana-extra')]

with patch('dhrishti.serving.active', lambda: cross_project):
    # Anchored at '-pyrepl-', so 'ramabana-extra' neither matches nor makes 'ramabana' ambiguous.
    test_eq(find_session('ramabana'), 'http://127.0.0.1:9004')

with patch('dhrishti.serving.active', lambda: by_cwd):
    # No name matches at all, so the whole-basename cwd fallback is what finds it.
    test_eq(find_session('leela'), 'http://127.0.0.1:9003')
    # A substring is not enough.
    test_fail(lambda: find_session('eela'), contains='no live dhrishti session')

# A scheme is taken as given and never touches the registry at all.
test_eq(find_session('http://127.0.0.1:9999'), 'http://127.0.0.1:9999')

### Colour

`rich` and `pygments` are already here for the transcript, so highlighting the input line and
the code cells costs no dependency. It is applied to the *rendering* only: `log_cell`, the
session notebook and what `/copy` yields all keep the plain source, because a highlighted
string is not code you can paste.

In [ ]:
#| export
def hl(src):
    "`src` as highlighted `Text` with `.plain == src`, or plain `Text` if pygments cannot lex it."
    src = str(src)
    if not src: return Text('')
    try:
        from rich.syntax import Syntax
        # `highlight` appends a newline; it is built for whole files
        out = Syntax(src, 'python', theme='gruvbox-dark').highlight(src)
        while out.plain.endswith('\n'): out.right_crop(1)
        return out if out.plain == src else Text(src)
    except Exception: return Text(src)

In [ ]:
t = hl('x = 1  # note')
test_eq(t.plain, 'x = 1  # note')            # the text is the text
assert t.spans                                # ...and it carries styles
test_eq(hl('').plain, '')

## The prompt, against a live kernel

The surface is `cli.Ui` and python mode is part of it; the cells below drive it from here
because this is where the kernel is. Agent mode is unchanged -- the same blocks, folding,
approvals, slash commands and status bar -- and python mode changes what a *line means*, not
what the terminal can do.

In [ ]:
from ramabana.testing import fake_agent
from ramabana.cli import CompletionMenu, Ui, run_turn
from teleprint.compositor import Compositor
from teleprint.keys import Key
from teleprint.testing import EmuTty        # not `pyghostty.EmuTty`; see nbs/05_cli.ipynb

tty = EmuTty(80, 40)   # tall enough for /help: `term.text()` is the active area, no scrollback
comp = Compositor(tty)
comp._register_signals = lambda: None    # nbdev runs this async cell on a worker thread
await comp.start()
agent, be = fake_agent(host=host, replies=['`owner` is a dict with one key.'])
ui = Ui(comp, agent)
ui.kernel = kernel          # the kernel above, so `enter_python` adopts it instead of starting a second
await ui.enter_python()
comp.on_key = ui.on_key
ui.paint()
test_eq(ui.mode, 'python')
assert 'python' in ui.status().plain

In [ ]:
# Switching is a slash command, so it costs no key and cannot be typed by accident.
ui.buf.insert('/agent'); test_eq(ui.submit(), None)
test_eq(ui.mode, 'agent')
assert ui.prompt().plain.startswith('▌')   # agent mode is the ordinary prompt, unchanged
ui.buf.insert('/py'); await ui.submit()
test_eq(ui.mode, 'python')

# `/help` is the one card, and it describes python mode too.
ui.buf.insert('/help'); ui.submit()
assert 'python' in tty.term.text()

# `/vars` is the session's own, and it reads what the kernel holds.
ui.buf.insert('/vars'); test_eq(ui.submit(), None)
assert 'owner' in ui.transcript.block_text(list(comp.blocks.values())[-1])

# Every other slash command is the base one's, so recall remembers the line and a completion
# menu armed while it was typed is dropped.
ui.buf.clear()
ui.buf.insert('/detach nothing.png')
ui.complete = CompletionMenu(ui.buf, ['/detach'], start=0, show=8)
test_eq(ui.submit(), None)
test_eq((ui.complete, ui.buf.text), (None, ''))
test_eq(ui.history[-1], '/detach nothing.png')

ui.buf.insert('recalled_py = 1')
await ui.submit()
test_eq(ui.history[-1], 'recalled_py = 1')     # a python line is a submitted line like any other
ui.recall(-1)
test_eq(ui.buf.text, 'recalled_py = 1')
ui.buf.clear()

In [ ]:
# Each output kind lands in the block that means it, so a traceback never reads as a reply.
ui.on_output({'output_type': 'stream', 'name': 'stdout', 'text': 'printed\n'})
ui.on_output({'output_type': 'execute_result', 'data': {'text/plain': '42'}})
ui.on_output({'output_type': 'error', 'ename': 'ValueError', 'evalue': 'bad', 'traceback': []})
kinds = [b.tag for b in comp.blocks.values()][-3:]
test_eq(kinds, ['note', 'reply', 'error'])

In [ ]:
# A python line executes in the owner's kernel and is logged; an agent line asks the model.
await ui.run_code('from_the_prompt = 99')
r = await kernel.execute('from_the_prompt')
test_eq(output_text(r.outputs), '99')
ui.mode = 'agent'
await run_turn(ui, 'what is in owner?')
assert be.sent, 'the turn reached the backend'
test_eq([c.cell_type for c in read_nb(host.agent_log).cells[-2:]], ['markdown', 'markdown'])
ui.mode = 'python'

# A host with nowhere to log still runs the cell and still leaves the prompt free.
_real_host = agent.host
try:
    for h in (nolog, dead_host):
        agent.host = h
        await ui.run_code('logless = 1')
        test_eq(ui.turn, None)
finally: agent.host = _real_host
test_eq(output_text((await kernel.execute('logless')).outputs), '1')

# And a log that raises cannot wedge the prompt either.
_real_log = host.log_cell
host.log_cell = lambda *a, **kw: 1/0
try: await ui.run_code('wedge_check = 1')
except ZeroDivisionError: pass
finally: host.log_cell = _real_log
test_eq(ui.turn, None)

In [ ]:
# Agent mode keeps the whole Ramabana surface: `/attach` and `@path` both work, and the picture
# itself reaches the backend as a content part.
pics = Path(tempfile.mkdtemp())
(pics / 'shot.png').write_bytes(b'\x89PNG\r\n\x1a\n' + b'0' * 64)
shot = pics / 'shot.png'

ui.mode = 'agent'
ui.buf.insert(f'/attach {shot}'); ui.submit()
assert 'shot.png' in ui.attach_row().plain
ui.buf.insert('what is in the picture?')
await ui.submit()
assert isinstance(be.sent[-1], list) and shot.read_bytes() in be.sent[-1]
test_eq(ui.attachments, [])                      # taken at the start of the turn, not left behind
ui.mode = 'python'

In [ ]:
# `@` inside a python line is Python's own syntax, not a file reference: `attach_refs` lives
# only in the agent-mode branch of `submit`.
leak = pics / '@leak.png'
leak.write_bytes(b'\x89PNG\r\n\x1a\n' + b'0' * 64)   # a real file, named to look like a reference

ui.mode = 'python'
ui.buf.insert("py_matmul = type('M', (), {'__matmul__': lambda self, o: 42})() @ 1")
await ui.submit()
r = await kernel.execute('py_matmul')
test_eq(output_text(r.outputs), '42')          # `@` ran as matrix multiplication, not a path
test_eq(ui.attachments, [])

ui.buf.insert(f'py_leak = 5  # not an attachment: @{leak}')
await ui.submit()
r = await kernel.execute('py_leak')
test_eq(output_text(r.outputs), '5')           # the line ran, `@leak.png` and all
test_eq(ui.attachments, [])                    # a python line never calls `attach_refs`

### Typing more than one line

`Ui` submits on `enter`, which is right for a sentence and wrong for a suite: `for i in
range(3):` is not a program yet. `codeop.compile_command` is the standard answer and tells the
three cases apart -- this compiles, this is unfinished, this will never compile -- so an
unfinished line grows the buffer and a broken one is reported at the prompt without a round
trip to the kernel.

An empty line submits what is pending, which is what every Python prompt does and what
fingers expect.

In [ ]:
#| export
def _compile_state(src, symbol):
    "`compile_command` as a (state, message) pair rather than three shapes of answer."
    try:
        code = codeop.compile_command(str(src), '<pyrepl>', symbol)
        return ('complete' if code is not None else 'incomplete'), ''
    except SyntaxError as e: return 'invalid', f'{e.msg} (line {e.lineno})'
    except Exception as e: return 'complete', agent_err(e)   # not ours to judge; let the kernel say

def _judge(src):
    "The prompt's verdict on `src`, and what objected when it was rejected."
    state, note = _compile_state(src, 'single')   # `'single'` is the question a REPL asks
    if state != 'invalid': return state, note
    return _compile_state(src, 'exec')            # ...but it also rejects two valid statements

def code_state(src):
    "Whether `src` is a finished statement, an unfinished one, or one that cannot compile."
    return _judge(src)[0]

def _syntax_note(src):
    "What the compile that rejected `src` objected to, in one line."
    return _judge(src)[1]

In [ ]:
test_eq(code_state('x = 1'), 'complete')
test_eq(code_state('for i in range(3):'), 'incomplete')
test_eq(code_state('x = ('), 'incomplete')
test_eq(code_state('x = )'), 'invalid')
test_eq(code_state(''), 'complete')
# A suite needs its blank line before it compiles, exactly as at a real prompt.
test_eq(code_state('for i in range(3):\n    print(i)'), 'incomplete')
test_eq(code_state('for i in range(3):\n    print(i)\n'), 'complete')

# Two statements are one paste, and `'single'` calls them invalid, so `'exec'` is asked too.
test_eq(code_state('x = 1\ny = 2\n'), 'complete')
test_eq(code_state('import os\nprint(os.getcwd())'), 'complete')
test_eq(code_state('import os\nfor i in range(2):\n    print(i)'), 'complete')

# The fallback swallows nothing: broken is broken under both, with the real objection said.
test_eq(code_state('x = 1\ny = )'), 'invalid')
assert 'unmatched' in _syntax_note('x = 1\ny = )')
assert 'multiple statements' not in _syntax_note('x = 1\ny = )')
test_eq(code_state('x = 1\ny = ('), 'incomplete')     # unfinished, not invalid: it grows

In [ ]:
# Multi-line editing, end to end: a suite grows until its blank line, the cursor tracks `CONT`,
# and what runs is what was typed.
await kernel.execute('multiline_ran = []')
ui.buf.clear()
ui.buf.insert('for i in range(3):')
ui.on_key(Key('enter'))
test_eq(ui.buf.text, 'for i in range(3):\n')       # grew, rather than being submitted
assert ui.turn is None
assert ui.CONT.strip() in tty.term.text()          # ...and the screen says so

ui.buf.insert('    multiline_ran.append(i)')
rows, (row, line_no, col) = ui.tail()
test_eq(row, len(rows) - 1)                        # the prompt is the last tail row...
test_eq(line_no, 1)                                # ...the cursor is on its second visual line...
test_eq(col, Text(ui.CONT + '    multiline_ran.append(i)').cell_len)   # ...at the column CONT implies
assert ui.CONT in ui.prompt().plain                # the row really is drawn with it
assert ui.prompt().spans                           # and coloured, which `tail` must not measure

ui.on_key(Key('enter'))                            # a suite still needs its blank line
test_eq(ui.buf.text, 'for i in range(3):\n    multiline_ran.append(i)\n')
out = ui.on_key(Key('enter'))                      # the blank line closes it
test_eq(ui.buf.text, '')
await out
r = await kernel.execute('multiline_ran')
test_eq(output_text(r.outputs), '[0, 1, 2]')       # the whole suite ran, its indentation intact
# The session notebook gets code that runs, not a rendering of it.
test_eq(read_nb(host.agent_log).cells[-1].source, 'for i in range(3):\n    multiline_ran.append(i)\n')

In [ ]:
# Invalid is reported at the prompt, with no round trip to the kernel.
ui.buf.clear()
ui.buf.insert('x = )')
ui.on_key(Key('enter'))
assert 'invalid' in tty.term.text() and ui.turn is None
ui.buf.clear()

# Agent mode is untouched: a question that happens to look like a suite is still a question.
ui.mode = 'agent'
ui.buf.insert('for the record:')
out = ui.on_key(Key('enter'))
test_eq(ui.buf.text, '')
if out is not None: await out
ui.mode = 'python'

In [ ]:
# A paste arrives as one buffer, so a two-statement snippet submits as it stands, and the kernel
# is the only witness that both statements ran.
ui.buf.clear()
ui.paste('pasted_a = 3\npasted_b = pasted_a * 2')
test_eq(code_state(ui.buf.text), 'complete')
out = ui.on_key(Key('enter'))
test_eq(ui.buf.text, '')                       # submitted, not reported as invalid
if out is not None: await out
r = await kernel.execute('pasted_b')
test_eq(output_text(r.outputs), '6')           # the second statement ran too, not just the first

In [ ]:
# Coloured on screen, plain on the way out: `/copy` and `block_text` both read `source`.
ui.buf.clear(); ui.buf.insert('df2 = df.copy()')
assert ui.prompt().spans                                   # coloured on screen...
blk = ui.say(hl('df2 = df.copy()'), 'user', source='df2 = df.copy()')
test_eq(ui.transcript.block_text(blk), 'df2 = df.copy()')  # ...and plain on the way out
ui.buf.clear()

### Getting back out of python mode

A slash line is a command, not code: `/agent` is invalid Python, so commands go to `submit` and
only code is compiled. Without that, python mode -- the default once you are in it -- has no
exit.

In [ ]:
def _line(u, s):
    for ch in s: u.on_key(Key(ch, char=ch))
    return u.on_key(Key('enter'))

test_eq(code_state('/agent'), 'invalid')     # it really is not Python
test_eq(_line(ui, '/agent'), None)
test_eq((ui.mode, ui.buf.text), ('agent', ''))
await _line(ui, '/python')
test_eq((ui.mode, ui.buf.text), ('python', ''))
test_eq(_line(ui, '/help'), None)             # and the commands still answer from python mode
test_eq(ui.buf.text, '')
assert 'python' in tty.term.text()
test_eq(_line(ui, '/quit'), 'quit')           # the escape hatch needs no keystroke, in either mode

# Tab is the same argument: `/mod` completes commands, not names.
ui.buf.clear(); ui.buf.insert('/mod')
test_eq(ui.on_key(Key('tab')), None)          # not a coroutine, so the kernel was never asked
test_eq(ui.buf.text, '/model')                # the common prefix of /model and /models
assert ui.complete is not None
ui.complete = None; ui.buf.clear()

# ...while a real suite is still compiled and still grows instead of submitting.
_line(ui, 'for i in range(3):')
test_eq(ui.buf.text, 'for i in range(3):\n')
ui.buf.clear()

The session does not hold the mouse, and `cli.amain` makes the same choice for the reason its
docstring gives: the main screen is the terminal's to select from, and `enter_transcript` borrows
the mouse only while the browsing view is up.

### Tab, against the live kernel

One match completes the buffer in place; several list, with the descriptions coming from
whatever the session is actually holding.

In [ ]:
await kernel.execute('completing = {"a": 1}')
ui.buf.clear(); ui.buf.insert('complet')
ui.buf.cursor = len(ui.buf.text)
await ui.complete_python()
test_eq(ui.buf.text, 'completing')             # one match completes in place
ui.buf.clear()

# Several matches list, and the ones that name live values say what they are.
await kernel.execute('twovals_a = 1\ntwovals_b = 2')
ui.buf.clear(); ui.buf.insert('twovals_'); ui.buf.cursor = len('twovals_')
await ui.complete_python()
assert ui.complete is not None, 'several matches arm the menu'
test_eq(sorted(ui.complete.matches), ['twovals_a', 'twovals_b'])
assert all('->' in d for d in ui.desc), ui.desc            # both are live, so both are described

# Several matches still extend the buffer as far as they agree; one character short of the
# common prefix is what reaches the reassignment.
ui.buf.clear(); ui.buf.insert('twoval'); ui.buf.cursor = len('twoval')
await ui.complete_python()
test_eq((ui.buf.text, ui.buf.cursor), ('twovals_', 8))
test_eq(len(ui.complete.matches), 2)                     # still a menu: the prefix is not a choice
ui.buf.clear()

In [ ]:
# `describe` is synchronous HTTP, so `complete_python` must keep it off the loop thread. The
# thread it runs on is recorded and compared with this test's own, which is the loop thread.
loop_thread = threading.current_thread()
seen_thread = {}
_real_describe = host.describe
def _watched_describe():
    seen_thread['t'] = threading.current_thread()
    return _real_describe()
host.describe = _watched_describe
try:
    await kernel.execute('threadcheck_a = 1\nthreadcheck_b = 2')
    ui.buf.clear(); ui.buf.insert('threadcheck_'); ui.buf.cursor = len('threadcheck_')
    await ui.complete_python()
finally:
    host.describe = _real_describe
assert seen_thread.get('t') is not None and seen_thread['t'] is not loop_thread
assert all('->' in d for d in ui.desc), ui.desc         # the off-thread lookup still landed
ui.buf.clear()

In [ ]:
# Typing is never gated by `self.turn`, so the buffer can move on while a `describe` lookup is
# in flight. A `describe` blocking on an `Event` puts the keystroke inside the await every run.
release = threading.Event()
def _blocked_describe():
    release.wait(5)
    return {'twovals_a': 'int', 'twovals_b': 'int'}
_real_describe = host.describe
host.describe = _blocked_describe
ui.complete, ui.desc = None, []   # clear what a previous test left, so the poll below is honest
try:
    await kernel.execute('twovals_a = 1\ntwovals_b = 2')
    ui.buf.clear(); ui.buf.insert('twovals_'); ui.buf.cursor = len('twovals_')
    task = asyncio.ensure_future(ui.complete_python())
    # the bare-names paint: `describe` is on its thread and blocked
    for _ in range(500):
        if ui.complete is not None: break
        await asyncio.sleep(0.01)
    assert ui.complete is not None and ui.desc == []   # names are up, types have not landed
    ui.on_key(Key('x', char='x'))                    # the user keeps typing while describe blocks
    test_eq(ui.buf.text, 'twovals_x')
    test_eq(ui.complete, None)                       # typing dropped the menu, synchronously
    release.set()                                    # let the blocked lookup return
    await task
finally:
    host.describe = _real_describe
    release.set()
test_eq((ui.complete, ui.desc), (None, []))   # the stale lookup never landed over the drop
test_eq(ui.buf.text, 'twovals_x')  # and the buffer kept the user's edit, untouched by the reply
ui.buf.clear()

## `/promote`

Promotion is the owner's act, so it is a command and never a tool: the model can ask for it in
words, and the person at the prompt is the one who does it. Own-kernel mode only -- attach mode
has no owner token to read, so the session belongs to whoever started it, and they promote from
their own surface.

In [ ]:
#| export
def promote(base, name):
    "Adopt an agent-layer variable into the owner namespace; own-kernel mode only."
    if not base: return 'no kernel of our own; promote from the surface that started the session'
    from dhrishti.serving import owner_token
    port = int(str(base).rsplit(':', 1)[-1])
    if (tok := owner_token(port)) is None: return f'no owner token for {base}; this session is not ours to change'
    try: result = _api(base, '/api/promote', {'accessor': json.dumps([str(name)]), 'token': tok})
    except Exception as exc: return agent_err(exc)
    return str(result.get('error') or f'promoted {name}')

In [ ]:
# Attach mode has no owner token to read, and the message says which condition it is.
test_eq(promote('', 'x'), 'no kernel of our own; promote from the surface that started the session')

In [ ]:
# `/promote` reaches the live ui through `submit`, and its blocking HTTP call runs off the loop
# thread. A stand-in, because the real call writes to the kernel this notebook is running in.
import ramabana.pyrepl as _pyr

seen = {}
def _fake_promote(base, name):
    seen.update(base=base, name=name, thread=threading.current_thread())
    return f'promoted {name}'
_real_promote, _pyr.promote = _pyr.promote, _fake_promote
try:
    ui.buf.insert('/promote answer_42')
    await ui.submit()
finally: _pyr.promote = _real_promote
test_eq((seen['base'], seen['name']), (kernel.base, 'answer_42'))
assert seen['thread'] is not threading.current_thread()   # off the loop thread
assert 'promoted answer_42' in tty.term.text()
test_eq(ui.buf.text, '')

# No argument is a usage line rather than "unknown command".
ui.buf.insert('/promote')
test_eq(ui.submit(), None)
assert 'usage: /promote' in tty.term.text()

### Attaching to the live session

The kernel this notebook started is itself a live dhrishti session, so `find_session` can
locate it by the name the registry gave it -- proof that attach works against a real server,
not a fake one. The name is `'<project>-pyrepl-<pid>'` (see `Kernel._bootstrap`), and the pid
keeps it unique next to any leftover from an earlier run.

In [ ]:
# The session this notebook's kernel started is a live one, so attach can find it by name; the
# pid in '<project>-pyrepl-<pid>' keeps that name unique.
from dhrishti.serving import active
own = next((e for e in active() if e.get('base') == kernel.base), None)
assert own, 'the kernel above is serving'
test_eq(find_session(own['name']), kernel.base)

# An attached host over the discovered base is an ordinary host -- no kernel involved. It is what
# `Ui.attach_session` builds.
attached = DhrishtiHost(['.'], find_session(own['name']), web=False, index=False)
test_eq(attached.kernel_kind, 'ipykernel')
assert 'owner' in attached.list_vars()

In [ ]:
# A polite shutdown has a deadline, and the rude kill is the fallback.
class _StuckKm:
    has_kernel = True
    def __init__(self): self.forced = None
    async def shutdown_kernel(self, now=False):
        self.forced = now
        if not now: await asyncio.sleep(30)      # never agrees, the way a busy kernel does not

stuck, km = Kernel('.'), _StuckKm()
stuck.km, stuck.kc = km, None
await stuck.shutdown(timeout=0.05)
test_eq(km.forced, True)                         # the deadline passed, so it stopped asking
test_eq(stuck.km, None)                          # ...and it finished rather than hanging
assert stuck.base is None

# A kernel that does agree is never killed rudely.
class _GoodKm(_StuckKm):
    async def shutdown_kernel(self, now=False): self.forced = now
good = _GoodKm(); ok = Kernel('.'); ok.km, ok.kc = good, None
await ok.shutdown(timeout=5)
test_eq(good.forced, False)

In [ ]:
await kernel.shutdown()
test_eq(kernel.alive, False)